# FluctlightDB — LongMemEval-S v2 (July 2026)

**Use this file:** `longmemeval_colab_v2.ipynb` (not the older `longmemeval_colab.ipynb`).

**Paper v2:** unified 500-question v4 retrieval + optional end-to-end QA.

Runs **session recall@8** with v4 harness:
- `--dual-key`, `--pref-facts-key`, `--query-expand`
- GPU embeddings via `multi-qa-mpnet-base-dot-v1`

## Profiles (`BENCH_PROFILE` in cell 2)
| Profile | Questions | Time (T4) |
|---------|-----------|-----------|
| **`v2`** | 500 retrieval + E2E | ~1.5 h retrieval + ~4–8 h E2E (Gemini free) |
| `full` | 500 retrieval | ~50–60 min |
| `preference` | 30 | ~5–15 min |
| `e2e` | 500 E2E only | ~4–8 h (Gemini free tier) |

**Paper v2 (recommended):** Colab → Secrets → **`GEMINI_API_KEY`**, set `BENCH_PROFILE = "v2"`, `E2E_LIMIT = 500`.

E2E uses **Google Gemini API** (`gemini-2.5-flash` reader + judge) via OpenAI-compatible chat API. Free tier caps at ~1M tokens/day — use **PayGo** for full 500 today.

Repo: [voxmastery/FluctlightDB](https://github.com/voxmastery/FluctlightDB)


In [1]:
# --- Configuration (edit before run) ---
REPO_URL = "https://github.com/voxmastery/FluctlightDB.git"
REPO_BRANCH = "main"

# "v2" = full 500 retrieval + e2e (paper v2) | "full" = retrieval only
# "preference" = 30 pref | "fast" = lexical | "e2e" = e2e only
BENCH_PROFILE = "e2e"  # v2 | full | preference | fast | e2e
LIMIT = 0  # 0 = all in profile; smoke test: 5
E2E_LIMIT = 500  # paper v2 full E2E (BENCH_PROFILE == "e2e" or "v2")
E2E_LLM_BACKEND = "openai"  # openai | gemini
E2E_PROFILE = "max"  # max = GPT-5 + CoT + top-50 reader | standard = gpt-4o
E2E_WORKERS = 4  # keep 1–2 if OpenAI rate-limits; 4 ok on Colab T4
TOP_K = 8  # session recall@8 metric (paper)
EMBED_MODEL = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
READER_MODEL = ""  # empty = profile default (gpt-5 for max)
JUDGE_MODEL = "gpt-4o-2024-08-06"  # official LongMemEval judge
# E2E: Colab Secrets → OPENAI_API_KEY or GEMINI_API_KEY
GEMINI_ENV_FILE = "/content/gemini.env"  # optional: GEMINI_API_KEY=...

PREF_FACTS_KEY = True
DUAL_KEY = True
QUERY_EXPAND = True

print("Profile:", BENCH_PROFILE, "limit:", LIMIT or "all", "e2e_limit:", E2E_LIMIT)
print("v4 flags: dual_key=", DUAL_KEY, "pref_facts_key=", PREF_FACTS_KEY, "query_expand=", QUERY_EXPAND)

Profile: e2e limit: all e2e_limit: 500
v4 flags: dual_key= True pref_facts_key= True query_expand= True


In [2]:
import subprocess, sys, os, shutil

def sh(cmd, check_returncode=True, **kw):
    print("Executing $", cmd) # Changed print to make it clearer what is a command and what is output
    current_env = os.environ.copy()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=current_env, **kw)
    if r.stdout:
        print("--- STDOUT ---\n", r.stdout)
    if r.stderr:
        print("--- STDERR ---\n", r.stderr)
    if r.returncode != 0 and check_returncode: # Only raise if check_returncode is True
        print(f"Command failed with exit code {r.returncode}")
        exit_code = r.returncode if r.returncode is not None else 1
        raise SystemExit(exit_code)
    return r # Return the result object for inspection if needed

# GPU check
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU runtime!")

sh("pip install -q maturin sentence-transformers datasets huggingface_hub PyYAML filelock fastapi uvicorn")

# Ensure .cargo and .rustup directories are completely removed before any rustup actions
home_dir = os.path.expanduser("~")
cargo_dir = os.path.join(home_dir, ".cargo")
rustup_dir = os.path.join(home_dir, ".rustup")

print("--- START RUST TOOLCHAIN CLEANUP & INSTALLATION ---")

if os.path.exists(cargo_dir):
    print(f"Removing existing {cargo_dir}...")
    shutil.rmtree(cargo_dir, ignore_errors=True)
if os.path.exists(rustup_dir):
    print(f"Removing existing {rustup_dir}...")
    shutil.rmtree(rustup_dir, ignore_errors=True)

# Clear .cargo from PATH if it was there previously
os.environ["PATH"] = ":".join([p for p in os.environ.get("PATH", "").split(":") if not p.startswith(cargo_dir)])
print("PATH after manual cleanup:", os.environ.get("PATH"))

print("Installing rustup and nightly toolchain as default...")
# Install Rustup and immediately set default toolchain to nightly
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --no-modify-path --default-toolchain nightly")

# Manually set PATH for the current Python process after rustup is installed
# Ensure it's prepended and unique
current_path_list = os.environ.get("PATH", "").split(":")
cargo_bin_path = os.path.join(home_dir, ".cargo", "bin")
if cargo_bin_path not in current_path_list:
    os.environ["PATH"] = cargo_bin_path + ":" + os.environ.get("PATH", "")
print("PATH after rustup install and manual update:", os.environ.get("PATH"))

# Ensure components are added to the installed toolchain
sh("rustup component add rustfmt --toolchain nightly")
sh("rustup component add clippy --toolchain nightly")

# Verify cargo is installed and in PATH after reinstallation
sh("which cargo") # Show which cargo is being used
sh("cargo --version")
sh("rustc --version")
print("--- END RUST TOOLCHAIN CLEANUP & INSTALLATION ---")


if not os.path.isdir("FluctlightDB"):
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} FluctlightDB")
else:
    sh("cd FluctlightDB && git pull --ff-only || true")

# Re-assert PATH after git pull in case it's changed, ensuring it's unique
current_path_list = os.environ.get("PATH", "").split(":")
cargo_bin_path = os.path.join(home_dir, ".cargo", "bin")
if cargo_bin_path not in current_path_list:
    os.environ["PATH"] = cargo_bin_path + ":" + os.environ.get("PATH", "")

# Verify contents of the directory before building
sh("ls -F FluctlightDB/crates/fluctlight-py/")

cargo_project_root = "FluctlightDB/crates/fluctlight-py"

# Remove Cargo.lock to ensure it's regenerated with the correct Cargo version
cargo_lock_path = os.path.join(cargo_project_root, "Cargo.lock") # Corrected path
if os.path.exists(cargo_lock_path):
    print(f"Removing existing {cargo_lock_path} to force regeneration...")
    os.remove(cargo_lock_path)

# Use `maturin build` and then explicitly check for the wheel file
# Add RUST_BACKTRACE=1 for verbose output in case of build failure
# Redirect output to a log file to capture full details if Colab truncates
maturin_log_filename_in_cwd = "maturin_build.log"
maturin_log_path = os.path.join(cargo_project_root, maturin_log_filename_in_cwd)
sh(f"RUST_BACKTRACE=1 maturin build --release > {maturin_log_filename_in_cwd} 2>&1", cwd=cargo_project_root, check_returncode=False)

# After maturin build, explicitly list content of target/wheels
target_wheels_dir = os.path.join("FluctlightDB", "target", "wheels") # Corrected path
if os.path.isdir(target_wheels_dir):
    print(f"Directory '{target_wheels_dir}' exists.")
    sh(f"ls -l {target_wheels_dir}") # list contents of wheels directory
    # Find the actual wheel file path
    wheel_files = [f for f in os.listdir(target_wheels_dir) if f.endswith(".whl")]
    if wheel_files:
        wheel_path = os.path.join(target_wheels_dir, wheel_files[0])
        print(f"Found wheel file: {wheel_path}")
        sh(f"pip install --force-reinstall {wheel_path}")
    else:
        raise SystemExit(f"No .whl files found in {target_wheels_dir} after maturin build.")
else:
    # Add more diagnostic info if wheels directory is missing
    print(f"Contents of {os.path.join(cargo_project_root, 'target')}:")
    sh(f"ls -F {os.path.join(cargo_project_root, 'target')}", check_returncode=False)
    print("Maturin build failed to produce wheels. Review output above for errors.")

    print(f"--- FULL MATURIN BUILD LOG ({maturin_log_path}) ---")
    if os.path.exists(maturin_log_path):
        with open(maturin_log_path, 'r') as f:
            print(f.read())
    else:
        print("Maturin log file not found.")
    print(f"--- END MATURIN BUILD LOG ---")

    raise SystemExit(f"Directory '{target_wheels_dir}' does NOT exist after maturin build. Maturin build failed to produce wheels.")

print("--- Installing Python SDK in editable mode ---")
sh("pip install -q -e FluctlightDB/sdks/python")

print("--- Diagnostic: Contents of FluctlightDB/sdks/python ---")
sh("ls -F FluctlightDB/sdks/python")

print("--- Diagnostic: Installed Python packages ---")
sh("pip list | grep fluctlightdb") # Check for fluctlightdb or similar

sys.path.insert(0, "FluctlightDB/sdks/python") # Explicitly add to sys.path

import fluctlightdb_native as _n
from fluctlightdb import connect_index
print("fluctlightdb native OK", _n.__version__ if hasattr(_n, '__version__') else "")

CUDA: True Tesla T4
Executing $ pip install -q maturin sentence-transformers datasets huggingface_hub PyYAML filelock fastapi uvicorn
--- STDOUT ---
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 110.3 MB/s eta 0:00:00

--- START RUST TOOLCHAIN CLEANUP & INSTALLATION ---
PATH after manual cleanup: /opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
Installing rustup and nightly toolchain as default...
Executing $ curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --no-modify-path --default-toolchain nightly
--- STDOUT ---
 
  nightly-x86_64-unknown-linux-gnu installed - rustc 1.98.0-nightly (c397dae80 2026-07-02)


Rust is installed now. Great!

To get started you need Cargo's bin directory ($HOME/.cargo/bin) in your PATH
environment variable. This has not been done automatically.

To configure your current shell, you need to source
the corresponding env file under $HOME/.c

In [3]:
from pathlib import Path
import json, urllib.request

DATA_PATH = Path("/content/longmemeval_s_cleaned.json")
if not DATA_PATH.is_file():
    url = "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json"
    print("Downloading", url, "(~500MB, one-time)...")
    urllib.request.urlretrieve(url, DATA_PATH)

data = json.loads(DATA_PATH.read_text())
print("Loaded", len(data), "questions from", DATA_PATH)

Loaded 500 questions from /content/longmemeval_s_cleaned.json


In [4]:
import sys
from typing import Optional

sys.path.insert(0, "FluctlightDB/benchmarks")
import longmemeval_bench as bench

class GpuEmbedCache:
    """In-process GPU embeddings (replaces HTTP embed sidecar)."""

    def __init__(self, model_name: str):
        from sentence_transformers import SentenceTransformer
        device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
        print(f"Loading {model_name} on {device}...")
        self.model = SentenceTransformer(model_name, device=device)
        self.cache: dict[str, list[float]] = {}
        self._requests = 0

    def _key(self, text: str) -> str:
        return (text or "").strip()[:4000]

    def embed_many(self, texts: list[str]) -> list[Optional[list[float]]]:
        out: list[Optional[list[float]]] = [None] * len(texts)
        missing_i, missing_t = [], []
        for i, t in enumerate(texts):
            k = self._key(t)
            if not k:
                continue
            if k in self.cache:
                out[i] = self.cache[k]
            else:
                missing_i.append(i)
                missing_t.append(k)
        if missing_t:
            unique = list(dict.fromkeys(missing_t))
            vecs = self.model.encode(
                unique, normalize_embeddings=True, batch_size=64, show_progress_bar=True
            )
            for t, v in zip(unique, vecs):
                self.cache[t] = v.tolist()
                self._requests += 1
            for i, t in zip(missing_i, missing_t):
                out[i] = self.cache.get(t)
        return out

    def embed_one(self, text: str) -> Optional[list[float]]:
        return self.embed_many([text])[0]

    def embed(self, text: str) -> Optional[list[float]]:
        return self.embed_one(text)

# Monkey-patch bench to use GPU embedder
bench.EmbedCache = GpuEmbedCache  # type: ignore
print("GpuEmbedCache ready")

GpuEmbedCache ready


In [5]:
import time
import os
import sys
import subprocess
from collections import defaultdict

# Cerebras API + local embed sidecar for E2E subprocess
def _load_llm_keys():
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GEMINI_KEY", "GEMINI_API_KEY_2"):
            try:
                os.environ["GEMINI_API_KEY"] = userdata.get(name)
                break
            except Exception:
                pass
        for name in ("OPENAI_API_KEY", "OPENAI_KEY"):
            try:
                os.environ["OPENAI_API_KEY"] = userdata.get(name)
                break
            except Exception:
                pass
    except Exception:
        pass
    if not os.environ.get("GEMINI_API_KEY") and Path(GEMINI_ENV_FILE).is_file():
        for line in Path(GEMINI_ENV_FILE).read_text().splitlines():
            if line.startswith("GEMINI_API_KEY="):
                os.environ["GEMINI_API_KEY"] = line.split("=", 1)[1].strip().strip('"').strip("'")
                break

_embed_proc = None

def _start_embed_server(port=8794):
    global _embed_proc
    os.environ["FLUCTLIGHT_EMBED_URL"] = f"http://127.0.0.1:{port}"
    os.environ["FLUCTLIGHT_EMBED_MODEL"] = EMBED_MODEL
    if _embed_proc is not None and _embed_proc.poll() is None:
        return
    import urllib.request
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
        return
    except Exception:
        pass
    _embed_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(port)],
        cwd="FluctlightDB/embed-server",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(90):
        if _embed_proc.poll() is not None:
            raise SystemExit("embed server failed to start")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            print("Embed sidecar ready on", port)
            return
        except Exception:
            time.sleep(2)
    raise SystemExit("embed server health check timeout")

def _e2e_env():
    _start_embed_server()
    _load_llm_keys()
    if E2E_LLM_BACKEND == "openai":
        if not os.environ.get("OPENAI_API_KEY"):
            raise SystemExit("Colab Secrets → add OPENAI_API_KEY (platform.openai.com)")
    elif E2E_LLM_BACKEND == "gemini":
        if not os.environ.get("GEMINI_API_KEY"):
            raise SystemExit("Colab Secrets → add GEMINI_API_KEY (aistudio.google.com/apikey)")
    env = {**os.environ, "PYTHONPATH": "FluctlightDB/sdks/python:FluctlightDB/benchmarks"}
    env["LONGMEMEVAL_LLM_BACKEND"] = E2E_LLM_BACKEND
    env["LONGMEMEVAL_E2E_WORKERS"] = str(E2E_WORKERS)
    env["LONGMEMEVAL_E2E_PROFILE"] = E2E_PROFILE
    env["LONGMEMEVAL_LLM_TIMEOUT"] = "300" if E2E_PROFILE == "max" else "180"
    return env

def _run_e2e_cmd(cmd, env):
    """Run e2e subprocess with live output; raise with stderr on failure."""
    print("Running e2e:", " ".join(cmd), flush=True)
    p = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f"E2E subprocess failed (exit {rc}). Check API key in Colab Secrets.")

def _e2e_cmd(limit, out_path, ckpt_path=None):
    cmd = [
        "python3", "FluctlightDB/benchmarks/longmemeval_e2e.py",
        "--data", str(DATA_PATH),
        "--limit", str(limit),
        "--top-k", str(TOP_K),
        "--dual-key", "--pref-facts-key", "--query-expand",
        "--llm-backend", E2E_LLM_BACKEND,
        "--e2e-profile", E2E_PROFILE,
        "--judge-model", JUDGE_MODEL,
        "--workers", str(E2E_WORKERS),
        "--json-out", str(out_path),
    ]
    if READER_MODEL:
        cmd.extend(["--reader-model", READER_MODEL])
    if ckpt_path:
        cmd.extend(["--checkpoint", str(ckpt_path)])
    return cmd

if BENCH_PROFILE == "e2e":
    limit = E2E_LIMIT if E2E_LIMIT > 0 else 500
    e2e_out = Path(f"/content/longmemeval_colab_e2e_{limit}.json")
    ckpt = Path(f"/content/longmemeval_colab_e2e_{limit}.checkpoint.jsonl")
    cmd = _e2e_cmd(limit, e2e_out, ckpt)
    _run_e2e_cmd(cmd, _e2e_env())
    print(e2e_out.read_text()[:4000])
    try:
        from google.colab import files
        files.download(str(e2e_out))
        files.download(str(ckpt))
    except ImportError:
        pass
    raise SystemExit(0)

items = data
_run_v2 = BENCH_PROFILE == "v2"
if BENCH_PROFILE in ("preference", "v2"):
    pass  # v2 uses all 500; preference filtered below
if BENCH_PROFILE == "preference":
    items = [x for x in items if x.get("question_type") == "single-session-preference"]
if LIMIT > 0:
    items = items[:LIMIT]

use_fast = BENCH_PROFILE == "fast"
dual_key = DUAL_KEY
query_expand = QUERY_EXPAND
pref_facts_key = PREF_FACTS_KEY and not use_fast  # pref-facts needs embed path

class NoEmbed:
    def __init__(self):
        self.cache = {}
        self._requests = 0
    def embed_many(self, texts):
        return [None] * len(texts)
    def embed(self, text):
        return None

embedder = NoEmbed() if use_fast else GpuEmbedCache(EMBED_MODEL)

print(f"Running {len(items)} questions | fast={use_fast} pref_facts_key={pref_facts_key}")

results = []
hits = 0
t0 = time.perf_counter()

for i, item in enumerate(items):
    row = bench.eval_one(
        item,
        mode="index",
        top_k=TOP_K,
        embedder=embedder,
        fast=use_fast,
        granularity="session",
        metric="session",
        query_expand=query_expand,
        dual_key=dual_key,
        pref_facts_key=pref_facts_key,
    )
    results.append(row)
    if row.get("hit"):
        hits += 1
    if (i + 1) % 5 == 0 or (i + 1) == len(items):
        print(f"[{i+1}/{len(items)}] session_recall@{TOP_K}={hits/(i+1):.1%} ({hits}/{i+1}) last_sec={row.get('sec')}")

wall = time.perf_counter() - t0
by_type = defaultdict(list)
for r in results:
    by_type[str(r.get("question_type") or "unknown")].append(bool(r.get("hit")))

pref_rate = by_type.get("single-session-preference")
pref_pct = round(sum(pref_rate) / len(pref_rate), 4) if pref_rate else None

report = {
    "benchmark": "longmemeval_s",
    "platform": "google_colab_gpu",
    "harness": "v4",
    "embed_model": EMBED_MODEL if not use_fast else None,
    "profile": BENCH_PROFILE,
    "granularity": "session",
    "metric": "session",
    "query_expand": query_expand,
    "dual_key": dual_key,
    "pref_facts_key": pref_facts_key,
    "top_k": TOP_K,
    "questions": len(results),
    "session_recall_at_k": round(hits / len(results), 4) if results else 0.0,
    "hits": f"{hits}/{len(results)}",
    "wall_s": round(wall, 1),
    "sec_per_question": round(wall / max(1, len(results)), 2),
    "embed_cache_size": len(getattr(embedder, "cache", {})),
    "embed_requests": getattr(embedder, "_requests", 0),
    "by_type": {k: round(sum(v) / len(v), 4) for k, v in sorted(by_type.items())},
}
if pref_pct is not None:
    report["preference_session_recall_at_k"] = pref_pct
    report["preference_target_90pct"] = pref_pct >= 0.90

out = {"summary": report, "results": results}
print("\n" + "=" * 60)
print("COPY EVERYTHING BELOW THIS LINE")
print("=" * 60)
print(json.dumps(out, indent=2))
print("=" * 60)

if pref_pct is not None:
    status = "MET ✓" if pref_pct >= 0.90 else "not yet"
    print(f"\nPreference session@{TOP_K}: {pref_pct:.1%} — 90% target {status}")

out_path = Path(f"/content/longmemeval_colab_{BENCH_PROFILE}_result.json")
if _run_v2:
    print("\n=== Paper v2: starting end-to-end QA (Gemini 2.5 Flash) ===")
    limit = E2E_LIMIT if E2E_LIMIT > 0 else 500
    e2e_out = Path(f"/content/longmemeval_colab_e2e_{limit}.json")
    ckpt = Path(f"/content/longmemeval_colab_e2e_{limit}.checkpoint.jsonl")
    cmd = _e2e_cmd(limit, e2e_out, ckpt)
    _run_e2e_cmd(cmd, _e2e_env())
    print(e2e_out.read_text()[:4000])
    try:
        from google.colab import files
        files.download(str(e2e_out))
        files.download(str(ckpt))
    except ImportError:
        pass

Embed sidecar ready on 8794
Running e2e: python3 FluctlightDB/benchmarks/longmemeval_e2e.py --data /content/longmemeval_s_cleaned.json --limit 500 --top-k 8 --dual-key --pref-facts-key --query-expand --llm-backend openai --reader-model gpt-4o-2024-08-06 --judge-model gpt-4o-mini --workers 4 --json-out /content/longmemeval_colab_e2e_500.json --checkpoint /content/longmemeval_colab_e2e_500.checkpoint.jsonl
LLM backend 'openai' failed smoke test: HTTP 401 https://api.openai.com/v1/chat/completions: {
  "error": {
    "message": "Incorrect API key provided: sk-proj-****************************e7YP. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}
Set Colab Secret (GEMINI_API_KEY or OPENAI_API_KEY) or pass --skip-smoke-test.


SystemExit: E2E subprocess failed (exit 1). Check API key in Colab Secrets.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Paste results back

Copy the JSON between the `====` lines and paste it in your FluctlightDB chat (or save as `benchmarks/results/longmemeval-preference-v4-mpnet-YYYY-MM-DD.json`).

### Profiles
| `BENCH_PROFILE` | What it runs |
|-----------------|--------------|
| `preference` | **30 preference questions** — v4 mpnet + dual-key + pref-facts-key + query-expand (~5–15 min) |
| `full` | 500 questions, same v4 flags |
| `fast` | 500 questions, lexical only (no embed; pref-facts disabled) |

### v4 flags (cell 2)
| Flag | Purpose |
|------|---------|
| `PREF_FACTS_KEY` | Third engram per session with extracted user facts (preference CP2) |
| `DUAL_KEY` | User-only index keys as second engram |
| `QUERY_EXPAND` | Multi-query + RRF merge for preference questions |

**Baselines:** lexical v4 = 86.7% (26/30); mpnet full without v4 = 76.7% (23/30). Target: **≥90%** on preference with mpnet + v4.

Set `LIMIT = 5` in cell 2 for a quick smoke test before the full 30.
